In [a previous notebook](https://www.kaggle.com/code/destbreso/which-language-does-your-agent-speak) I showed how to classify any agent in this competition from a handful of its public replays, by where its cross-game language melts: **SCRIPT** (a frozen plan, recited every game), **BRANCHER** (a repeated trunk that forks on something observed), **SCHEDULER** (a plan composed fresh per game), and a **MIXED** bucket between them, with a complexity index ordering the whole field from reciting to composing.

Of those classes, the BRANCHER is the least complex machine that actually decides anything: a script would need no observations at all, and a scheduler is a whole planner, but a BRANCHER is a plan with a few readable choice points. Which invites going one step further, and that is the question of this page: **can a BRANCHER be reverse engineered from its replays alone?** Not classified, read: at which turns does its plan fork, what observable does each fork condition on, and does the deduced decision tree predict games the agent has never played?

The answer, measured on five agents, two where I know the ground truth and three read blind, is yes: **123 of 126 held-out branch predictions correct, 97.6 percent**, with the known trees recovered fork for fork and variable for variable. That is the 97 percent of the title, and its exact scope so the number cannot oversell: correct held-out predictions of WHICH BRANCH the agent takes, at the forks with enough episodes on each side to test, every such fork of the five-agent study counted and none cherry-picked; it is not a claim about every turn of play, section 7's later case keeps its own ledger, and section 9 lists what stays out of reach.

The plan: first the fair question, whether BRANCHERs even matter at the top (section 1), then the method in three steps (section 2) and its complete code (section 3), a live read of my own agent whose true tree I know by construction (section 4), a historical agent I had reverse engineered by hand months ago, recovered by machine at exact precision (section 5), three top-15 agents read blind with leave-one-out as the judge (section 6), a public ground truth that walked in while this page was being written, an author who published his decision table outright, and what the instrument found when it took that exam (section 7), what the trees have in common (section 8), and the limits (section 9).

The rules of the series hold: every number is computed by the page or carried with its date, the method is complete enough to rerun on any submission id, and what the instrument cannot decide is said out loud rather than smoothed over.

## 1. Are there BRANCHERs at the top?

Before reading anyone's tree, the fair question is whether the class matters at all: perhaps BRANCHERs are a mid-table species and the top belongs to the composers. The census from the previous notebook, the top fifteen teams fingerprinted from their public replays with ranks read on 2026-09-04, says otherwise, and the chart below redraws it for this page's question:

* The one pure BRANCHER in that top 15 was **the rank 1 team of the pull**. The least complex machine that decides anything was, that day, beating everyone.
* Six more of the fifteen are MIXED, and the three this page reads blind in section 6, ranks 2, 5 and 9 at the pull, all carry an early trunk, which is exactly the structure this method needs.
* The historical ground truth of section 5, カワシギ, was one of the strong agents of its era and is a pure BRANCHER on cash.

So the class has not merely appeared at the top, it has held rank 1. A small tree over the right observables is evidently enough to compete with full per-game planners, and that is what makes the question of this page worth asking: a machine that simple, doing that well, should be readable.

In [ ]:
# The top 15 as fingerprinted on 2026-09-04 (ranks and ratings of that read),
# from the census of the previous notebook, redrawn for this page's question.
import json as _json
import matplotlib.pyplot as plt

CENSUS15 = _json.loads(r'''[{"rank": 1, "team": "keiz", "rating": 3054.1, "cls": "BRANCHER"}, {"rank": 2, "team": "Jesse Bullard", "rating": 2987.8, "cls": "MIXED"}, {"rank": 3, "team": "Andrey Tikhomirov", "rating": 2931.7, "cls": "SCHEDULER"}, {"rank": 4, "team": "kwa", "rating": 2897.4, "cls": "SCHEDULER"}, {"rank": 5, "team": "MtN", "rating": 2896.0, "cls": "MIXED"}, {"rank": 6, "team": "OceanMix", "rating": 2873.1, "cls": "SCHEDULER"}, {"rank": 7, "team": "CemBas", "rating": 2856.7, "cls": "MIXED"}, {"rank": 8, "team": "b13902103_yuhung94", "rating": 2840.9, "cls": "SCHEDULER"}, {"rank": 9, "team": "Giulio Ravasio", "rating": 2838.6, "cls": "MIXED"}, {"rank": 10, "team": "Scott Willis", "rating": 2836.2, "cls": "SCRIPT"}, {"rank": 11, "team": "peikopon", "rating": 2835.5, "cls": "MIXED"}, {"rank": 12, "team": "Zhongyi Dai", "rating": 2831.6, "cls": "SCHEDULER"}, {"rank": 13, "team": "Dmytro Maliarenko", "rating": 2828.1, "cls": "SCRIPT"}, {"rank": 14, "team": "Atakan Aldemir", "rating": 2819.1, "cls": "SCHEDULER"}, {"rank": 15, "team": "LagrangianLocomotive", "rating": 2810.4, "cls": "MIXED"}]''')
CLASS_COLOR = {"SCRIPT": "#c44536", "BRANCHER": "#b45309",
               "SCHEDULER": "#0f172a", "MIXED": "#8a8f98"}
BLIND = {2, 5, 9}   # read blind in section 6

fig, ax = plt.subplots(figsize=(9.0, 3.4))
for c in CENSUS15:
    ax.scatter(c["rank"], c["rating"], s=90, c=CLASS_COLOR[c["cls"]], zorder=3)
    if c["cls"] == "BRANCHER":
        ax.annotate(f"{c['team']}: the BRANCHER, rank 1 at the pull", (c["rank"], c["rating"]),
                    textcoords="offset points", xytext=(12, -16), fontsize=8, color="#b45309")
    if c["rank"] in BLIND:
        ax.scatter(c["rank"], c["rating"], s=260, facecolors="none",
                   edgecolors="#333", linewidths=1.1, zorder=2)
        ax.annotate("read blind below", (c["rank"], c["rating"]),
                    textcoords="offset points", xytext=(0, -16), ha="center", fontsize=6.5, color="#333")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=v, label=k) for k, v in CLASS_COLOR.items()],
          fontsize=7, frameon=False, ncol=4, loc="upper right")
ax.set_xlabel("rank as read 2026-09-04")
ax.set_ylabel("rating at that read")
ax.set_xticks(range(1, 16))
ax.set_title("the top 15 by architecture: one BRANCHER, and it sat at rank 1", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

import collections
print(dict(collections.Counter(c["cls"] for c in CENSUS15)))


## 2. The method: where, what, and does it predict

A BRANCHER repeats a trunk and then forks on something it observes. Reading its tree takes three steps, each with its own failure mode and its own control:

**Where are the forks.** Play the agent's episodes forward together and watch the number of distinct action-prefix classes: every turn where one group of episodes stops agreeing and splits is a fork. This is pure action analysis, no engine involved, and it is the same machinery my language notebook validated blind by recovering seven hand-measured branch turns of a known agent at exact precision with zero false alarms.

**What does each fork read.** At a fork the episodes split into groups; the question is which observable separates the groups. I reconstruct the exact observation the agent saw at the fork by replaying both recorded action streams through the official engine from the recorded seed (a step that must reproduce the recorded final banks to the dollar, or the episode is dropped), then run a decision-stump search over a menu of features: own money, rival money, every product price, own shed, unlocked shops, each at the fork turn, one turn earlier, and one day earlier. A stump that separates the groups perfectly is a candidate rule.

**Does the tree predict.** With few episodes, some feature will separate any split by luck. Two controls: a **permutation p-value** (shuffle the branch labels, count how often some feature separates that well by chance), and **leave-one-out prediction**: refit the stump without one episode, then predict which branch that episode took from its own observation. The LOO count is the number that decides whether a tree is real.

And one honesty rule the first ground truth will demonstrate: **not every fork is a decision.** A weed spawning on a tile reschedules a farmhand and produces a fork that no observable explains; the permutation test is what keeps such noise out of the tree.

## 3. The instrument, complete

Three cells: the fork detector with the stump search and its two controls (pure standard library), the replay puller, and the observation reconstruction through the official engine with the bank-reproduction gate. Point `DEMO_SUBMISSION` further down at any submission id, your own included.

In [ ]:
# Step 1 and step 3: forks, stumps, permutation, leave-one-out. Pure stdlib.
import collections, itertools, json, random
import statistics as st

PASS = {"farmer": ["PASS"], "hands": [], "market": []}

def canon(a):
    return json.dumps(a or PASS, sort_keys=True, separators=(",", ":"))

def chan_of(a, b):
    a, b = a or PASS, b or PASS
    plan = (a.get("farmer"), a.get("hands")) != (b.get("farmer"), b.get("hands"))
    mkt = (a.get("market") or []) != (b.get("market") or [])
    return "plan+market" if plan and mkt else ("plan" if plan else "market")

def find_forks(eps, min_group=4):
    """walk turns; a fork is where one prefix-group splits into >= 2 action-groups"""
    forks = []
    def walk(idx, t0):
        for t in range(t0, 719):
            groups = collections.defaultdict(list)
            for i in idx:
                groups[canon(eps[i]["acts"][t])].append(i)
            if len(groups) > 1:
                kids = sorted(groups.values(), key=len, reverse=True)
                ex = [eps[kids[0][0]]["acts"][t], eps[kids[1][0]]["acts"][t]]
                forks.append({"turn": t, "parent": sorted(idx),
                              "children": [sorted(k) for k in kids],
                              "channel": chan_of(ex[0], ex[1])})
                for k in kids:
                    if len(k) >= min_group:
                        walk(k, t + 1)
                return
    walk(list(range(len(eps))), 0)
    forks.sort(key=lambda f: (f["turn"], -len(f["parent"])))
    return forks

def _pair_agree(pred, labels):
    idx = sorted(labels); tot = same = 0
    for a, b in itertools.combinations(idx, 2):
        tot += 1
        if (pred[a] == pred[b]) == (labels[a] == labels[b]):
            same += 1
    return same / tot if tot else 0.0

def try_stump(vals, labels):
    idx = sorted(labels)
    if any(vals[i] is None for i in idx):
        return None
    part = {}
    for i in idx:
        part.setdefault(vals[i], []).append(i)
    best = ("eq", None, _pair_agree({i: k for k, v in enumerate(part.values()) for i in v}, labels))
    if len(set(labels.values())) == 2 and all(isinstance(vals[i], (int, float)) for i in idx):
        order = sorted(idx, key=lambda i: vals[i])
        for cut in range(1, len(order)):
            lo, hi = order[:cut], order[cut:]
            if vals[lo[-1]] == vals[hi[0]]:
                continue
            sc = _pair_agree({i: 0 for i in lo} | {i: 1 for i in hi}, labels)
            if sc > best[2]:
                best = ("thr", (vals[lo[-1]] + vals[hi[0]]) / 2, sc)
    return best

def perm_p(vals, labels, observed, n=2000, rng=None):
    rng = rng or random.Random(7)
    idx = sorted(labels); labs = [labels[i] for i in idx]; hit = 0
    for _ in range(n):
        rng.shuffle(labs)
        stp = try_stump(vals, dict(zip(idx, labs)))
        if stp and stp[2] >= observed - 1e-9:
            hit += 1
    return (hit + 1) / (n + 1)

def label_of(fork):
    return {i: ci for ci, kid in enumerate(fork["children"]) for i in kid}

def feature_menu(feats, e_idx, t):
    out = []
    for dt, tag in ((0, ""), (-1, "@t-1"), (-24, "@t-24")):
        tt = t + dt
        if tt < 0 or tt >= len(feats[e_idx[0]]):
            continue
        for k in feats[e_idx[0]][tt]:
            out.append((k + tag, {i: feats[i][tt][k] for i in e_idx}))
    return out

def explain_fork(fork, feats, rng=None):
    labels = label_of(fork); idx = sorted(labels); cands = []
    for name, vals in feature_menu(feats, idx, fork["turn"]):
        stp = try_stump(vals, labels)
        if stp:
            cands.append((stp[2], name, stp, vals))
    if not cands:
        return None
    cands.sort(key=lambda x: -x[0])
    sc, name, stp, vals = cands[0]
    ties = sorted(c[1] for c in cands if c[0] >= sc - 1e-9)
    return {"feature": name, "kind": stp[0], "rule": stp[1], "score": round(sc, 3),
            "perm_p": round(perm_p(vals, labels, sc, rng=rng), 4), "ties": ties}

def loo_fork(fork, feats):
    labels = label_of(fork); idx = sorted(labels)
    if min(len(k) for k in fork["children"]) < 2 or len(idx) < 4:
        return None
    hits = tot = 0
    for held in idx:
        rest = {i: labels[i] for i in idx if i != held}
        best = None
        for name, vals in feature_menu(feats, idx, fork["turn"]):
            stp = try_stump({i: vals[i] for i in rest}, rest)
            if stp and (best is None or stp[2] > best[0]):
                best = (stp[2], name, stp, vals)
        if best is None or best[3][held] is None:
            continue
        _, name, stp, vals = best
        v = vals[held]
        if stp[0] == "eq":
            same = [i for i in rest if vals[i] == v]
            pred = collections.Counter(rest[i] for i in same).most_common(1)[0][0] if same else None
        else:
            lo = [i for i in rest if vals[i] <= stp[1]]
            grp = lo if v <= stp[1] else [i for i in rest if i not in lo]
            pred = collections.Counter(rest[i] for i in grp).most_common(1)[0][0] if grp else None
        if pred is not None:
            tot += 1
            hits += (pred == labels[held])
    return (hits, tot)

print("fork machinery loaded")


In [ ]:
# Step 0: pull the newest replays of one submission, politely.
import time
import requests

LIST_URL = "https://www.kaggle.com/api/i/competitions.EpisodeService/ListEpisodes"
REPLAY_URL = "https://www.kaggleusercontent.com/episodes/{id}.json"

def pull_episodes(submission_id, max_eps=14, pace=1.5, episode_ids=None):
    eps = requests.post(LIST_URL, json={"submissionId": int(submission_id)},
                        timeout=30).json().get("episodes", [])
    eps = [e for e in eps if e.get("state") == "COMPLETED"]
    eps.sort(key=lambda e: e.get("endTime") or "", reverse=True)
    if episode_ids:                       # pin: the page reproduces itself
        wanted = {int(x) for x in episode_ids}
        eps = [e for e in eps if e["id"] in wanted]
    out = []
    for e in eps:
        if len(out) >= max_eps:
            break
        r = requests.get(REPLAY_URL.format(id=e["id"]), timeout=120)
        time.sleep(pace)
        if not r.ok:
            continue
        d = r.json()
        info = d.get("info") or {}
        steps = d.get("steps") or []
        seed = info.get("seed")
        if len(steps) < 700 or seed is None:
            continue
        seat = next(((a.get("index") or 0) for a in e["agents"]
                     if a.get("submissionId") == int(submission_id)), None)
        if seat is None:
            continue
        # a replay stores the action decided at engine turn t at steps[t + 1]
        acts = [(steps[t + 1][seat].get("action") if t + 1 < len(steps) else None) or PASS
                for t in range(719)]
        opp = [(steps[t + 1][1 - seat].get("action") if t + 1 < len(steps) else None) or PASS
               for t in range(719)]
        rw = [s.get("reward") for s in steps[-1]]
        out.append({"episode": e["id"], "seed": int(seed), "seat": seat, "acts": acts,
                    "opp_acts": opp, "bank": rw[seat]})
    return out

print("puller ready")


In [ ]:
# Step 2: reconstruct what the agent SAW, by replaying both recorded streams
# through the official engine from the recorded seed. These episodes were played
# on engine 1.32.7; the platform image may carry an older build, so the right
# version is installed (checked WITHOUT importing, so no half-loaded package)
# and the replay runs in a clean subprocess.
import importlib.metadata as _im
import json as _json2, pathlib as _pl, subprocess, sys

try:
    _need = tuple(map(int, _im.version("kaggle-environments").split(".")[:3])) < (1, 32, 7)
except Exception:
    _need = True
if _need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "kaggle-environments==1.32.7"], check=True)
ENGINE_VERSION = _im.version("kaggle-environments")

_WORKER = r"""
import json, sys
from kaggle_environments import make

PASS = {"farmer": ["PASS"], "hands": [], "market": []}
PRODUCTS = ("CARROT", "EGG", "FERTILIZER", "MELON", "MILK", "STRAWBERRY",
            "TOMATO", "WHEAT", "WOOL")

def features_of(e):
    env = make("kaggriculture", configuration={"seed": int(e["seed"])})
    env.reset(2)
    seat = e["seat"]
    rows = []
    for t in range(719):
        o = env.state[seat].observation
        farms = o["farms"]
        pr = (o.get("market") or {}).get("prices") or {}
        shed = (o.get("private") or {}).get("shed") or {}
        row = {"money": farms[seat].get("money"),
               "opp_money": farms[1 - seat].get("money"),
               "unlocked": "+".join((o.get("town") or {}).get("unlocked_shops") or [])}
        for p in PRODUCTS:
            row[f"price_{p}"] = pr.get(p)
            row[f"shed_{p}"] = shed.get(p)
        rows.append(row)
        aa, bb = e["acts"][t] or PASS, e["opp_acts"][t] or PASS
        env.step([aa, bb] if seat == 0 else [bb, aa])
    mine = env.state[seat].reward
    drift = None if e.get("bank") is None else mine - e["bank"]
    return rows, drift

eps = json.load(open(sys.argv[1]))
out = [features_of(e) for e in eps]
json.dump(out, open(sys.argv[2], "w"))
"""

def reconstruct(eps):
    # all episodes' feature trajectories plus bank drifts, via a clean interpreter
    w = _pl.Path("_replay_worker.py"); w.write_text(_WORKER)
    fi, fo = _pl.Path("_eps_in.json"), _pl.Path("_feats_out.json")
    fi.write_text(_json2.dumps(eps))
    subprocess.run([sys.executable, str(w), str(fi), str(fo)], check=True)
    return _json2.loads(fo.read_text())

def analyze(eps, min_group=4, label="", tol=0.01):
    feats, kept, drifts = [], [], []
    for e, (rows, drift) in zip(eps, reconstruct(eps)):
        if drift is None:
            continue
        if abs(drift) < 0.5 or abs(drift) <= tol * max(1.0, abs(e["bank"])):
            kept.append(e); feats.append(rows); drifts.append(drift)
    exact = sum(1 for x in drifts if abs(x) < 0.5)
    msg = f"{label}: engine {ENGINE_VERSION}; {len(kept)}/{len(eps)} episodes usable, {exact} to the dollar"
    if len(kept) > exact:
        msg += f", worst tolerated drift {max(abs(x) for x in drifts):,.0f}"
    print(msg)
    if len(kept) > exact:
        print("NOTE: these recordings come from a different engine build, so they replay within "
              "tolerance rather than exactly. Fork turns are recorded actions and stay exact; "
              "the reconstructed features (money, prices) carry drift of the order above.")
    if len(kept) < 6:
        print("fewer than 6: a tree inferred from that is a guess. Stopping.")
        return None
    forks = find_forks(kept, min_group)
    rng = random.Random(11)
    agg_h = agg_t = 0
    print(f"{'turn':>5} {'day':>4} {'n':>3} {'split':<9} {'channel':<12} "
          f"{'feature':<18} {'score':>6} {'p':>7} {'LOO':>6}")
    for fk in forks:
        ex = explain_fork(fk, feats, rng)
        loo = loo_fork(fk, feats)
        if loo:
            agg_h += loo[0]; agg_t += loo[1]
        sizes = "/".join(str(len(k)) for k in fk["children"])
        print(f"{fk['turn']:>5} {fk['turn'] // 24:>4} {len(fk['parent']):>3} {sizes:<9} "
              f"{fk['channel']:<12} {(ex or {}).get('feature', '?'):<18} "
              f"{(ex or {}).get('score', '-'):>6} {(ex or {}).get('perm_p', '-'):>7} "
              f"{f'{loo[0]}/{loo[1]}' if loo else '-':>6}")
    if agg_t:
        print(f"\nleave-one-out over all testable forks: {agg_h}/{agg_t} = {agg_h / agg_t:.1%}")
    return forks

def read_tree(submission_id, max_eps=14, min_group=4):
    return analyze(pull_episodes(submission_id, max_eps), min_group, f"submission {submission_id}")

print(f"observation reconstruction ready (official engine {ENGINE_VERSION}, clean-subprocess replay)")


## 4. Ground truth 1: my own agent, read live

The best first test of a reverse-engineering instrument is an agent whose insides I know by construction. My current submission plays a fixed farm plan on a public chassis that routes its MARKET orders on the shops the town unlocks (credited and documented layer by layer in my finance notebook), plus a thin cash-conditioned finance layer of my own. So the true tree is: one big routing fork when the day-6 shop is known, market channel only, and small cash-conditioned sell edits that fire rarely.

The cell below runs the whole pipeline on that submission, pinned to the fourteen episodes of my reference read of 2026-09-05 so the page reproduces its own numbers (use `read_tree(...)` with any submission id, yours included, to read an agent's newest games live). The replay gate prints the engine version it runs on and how exactly each episode reproduces: on kaggle-environments 1.32.7 all fourteen replay to the dollar; on a different build they replay within a stated tolerance, the note says so, and the fork turns, being recorded actions, are exact either way. The read was exactly the build sheet:

* **The routing fork, found**: t=150 (day 6, hour 6), market channel, separated 1.00 by a product price that indexes the unlocked shop, permutation p = 0.0005, leave-one-out 13 of 13. The engine unlocks the second shop at day 6 and the agent's sell routes split right there.
* **The finance layer, seen but honestly unresolved**: money-separated micro-forks at t=206 and t=265, exactly where a cash-conditioned layer edits sells, but at 14 episodes the permutation test refuses to call them (p 0.2 to 0.5). A micro-fork needs a bigger sample than a routing fork, and the instrument says so instead of overclaiming.
* **The noise exhibit**: a plan fork at t=21 splitting one episode from the other thirteen, best feature p = 0.082. That episode's deviation is one farmhand collecting fertilizer a turn early because a weed rescheduled its CARE: environmental noise, correctly left out of the tree.

Leave-one-out over all testable forks: **18 of 18**.

An unpinned rerun over a fresher episode mix later that day added one more build-sheet hit: a plan fork at exactly t=360, which is the turn one of my edit layers un-gates, again below significance at four episodes. Different episode mixes surface different micro-forks; the routing fork and the money family are what reproduce every time.

In [ ]:
# My live agent (submission 56035588), the reference read of 2026-09-05,
# embedded as its fourteen public streams so this page reproduces its own
# numbers; observations are reconstructed here by the official engine.
import base64 as _b64, zlib as _zl, json as _j
REFERENCE = _j.loads(_zl.decompress(_b64.b64decode(
"...")).decode())
print(f"reference streams loaded: {len(REFERENCE)} episodes")


In [ ]:
forks_demo = analyze(REFERENCE, label="my agent 56035588 (14 reference episodes)")

# to read ANY agent live instead, yours or a rival, pull its newest games:
# read_tree(YOUR_SUBMISSION_ID, max_eps=14)


## 5. Ground truth 2: a hand-derived tree, recovered by machine

Months ago I reverse engineered カワシギ's submission 55425101 by hand, probing branch by branch: seven turns where its plan forks, and the variable it reads, its own cash. That work took days. The instrument gets the same tree in about a minute.

On the same eight public episodes: **all seven fork turns recovered exactly** (engine turns 72, 113, 177, 264, 312, 357, 384), the root fork's best separator is **money** (permutation p = 0.028, and it stays the selected variable down the tree), and leave-one-out on the testable forks is **12 of 12**. The chart below is the recovered tree as a timeline.

Two honest notes. First, at eight episodes the deep forks split two episodes into one and one, which no statistic can bless; the tree's depth is limited by the sample, and the instrument reports those forks with p = 1.0 rather than hiding them. Second, these are recordings from an earlier engine version: they replay within a few hundred dollars rather than to the dollar, so the reconstructed money values carry drift of that order; the fork turns, being recorded actions, are exact. The tolerance and the warning are part of the tool.

In [ ]:
# The recovered tree of カワシギ 55425101, embedded with its date (2026-09-05):
# seven known fork turns, all found at delta zero, money the root variable.
import json as _json
KAWASHIGI = _json.loads(r'''[{"turn": 72, "channel": "market", "n": 8, "split": "6/2", "feature": "money", "score": 1.0, "perm_p": 0.028, "loo": "8/8"}, {"turn": 113, "channel": "plan", "n": 2, "split": "1/1", "feature": "unlocked", "score": 1.0, "perm_p": 1.0, "loo": null}, {"turn": 177, "channel": "plan", "n": 6, "split": "5/1", "feature": "opp_money", "score": 1.0, "perm_p": 0.1609, "loo": null}, {"turn": 264, "channel": "market", "n": 5, "split": "4/1", "feature": "money", "score": 1.0, "perm_p": 0.4178, "loo": null}, {"turn": 312, "channel": "market", "n": 4, "split": "2/2", "feature": "money", "score": 1.0, "perm_p": 0.3158, "loo": "4/4"}, {"turn": 357, "channel": "market", "n": 2, "split": "1/1", "feature": "money", "score": 1.0, "perm_p": 1.0, "loo": null}, {"turn": 384, "channel": "market", "n": 2, "split": "1/1", "feature": "money", "score": 1.0, "perm_p": 1.0, "loo": null}]''')

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9.2, 2.8))
known = [72, 113, 177, 264, 312, 357, 384]
for t in known:
    ax.axvline(t, color="#c44536", ls="--", lw=0.9)
sig = [f for f in KAWASHIGI if f.get("perm_p") is not None]
ax.scatter([f["turn"] for f in sig], [1] * len(sig),
           s=[40 + 12 * f["n"] for f in sig],
           c=["#b45309" if (f.get("perm_p") or 1) <= 0.05 else "#8a8f98" for f in sig],  # amber: its variable is cash
           zorder=3)
for f in sig:
    ax.annotate(f"n={f['n']}\np={f.get('perm_p')}", (f["turn"], 1), textcoords="offset points",
                xytext=(0, 12), ha="center", fontsize=6.5)
ax.set_yticks([])
ax.set_xlabel("engine turn (dashed red: the seven hand-measured branch turns)")
ax.set_title("kawashigi 55425101: machine-recovered forks land exactly on the hand-measured tree",
             fontsize=9, loc="left")
ax.set_xlim(0, 450)
plt.tight_layout(); plt.show()

_root = max(KAWASHIGI, key=lambda f: f["n"])
_h = _t = 0
for f in KAWASHIGI:
    if f.get("loo"):
        a, bb = map(int, f["loo"].split("/")); _h += a; _t += bb
print(f"root fork t={_root['turn']}: split {_root['split']}, best separator "
      f"{(_root['feature'] or '?').upper()}, p = {_root['perm_p']}, LOO {_root['loo']}")
print(f"leave-one-out on all testable forks: {_h}/{_t}")


## 6. Three top-15 agents, read blind

Ground truths calibrate; the real test is agents I know nothing about beyond their public replays, where leave-one-out is the only judge. Three current top-15 teams whose fingerprints showed a trunk, read from their ten to nineteen newest verified episodes:

| agent | episodes | significant forks (p <= 0.05) | LOO |
|---|---|---|---|
| MtN (rank 5 at the 2026-09-04 pull) | 19 | world forks at t=80 and t=144/150 (p <= 0.011), a CASH fork at t=155 (p = 0.037) | **73/74 = 98.6 %** |
| Giulio Ravasio (rank 9) | 12 | a world fork at t=150 (p = 0.011) | 16/18 = 88.9 % |
| Jesse Bullard (rank 2) | 10 | few testable forks: a scripted plan under a market layer | 4/4 |

MtN is the page's best blind read: a full two-level world router with a cash branch on top, every level predicting held-out games at 19 of 19 or 16 of 17. Its tree is, functionally, a build sheet, and the cell below draws it as one: the whole inferred hierarchy, nodes at their fork turns, leaves at the groups of games each path leads to.

In [ ]:
# The page finally draws a TREE: MtN's, from the full inferred hierarchy
# (embedded 2026-09-05), nodes at their fork turns, leaves = groups of games.
MTN = _json.loads(r'''[{"turn": 80, "channel": "market", "split": "17/2", "n": 19, "parent": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18], "children": [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18], [10, 11]], "feature": "price_WOOL", "kind": "eq", "rule": null, "score": 1.0, "perm_p": 0.007, "ties": ["price_WOOL", "price_WOOL@t-1"], "loo": "19/19"}, {"turn": 144, "channel": "market", "split": "13/4", "n": 17, "parent": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18], "children": [[4, 5, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18], [0, 1, 2, 3]], "feature": "price_WHEAT", "kind": "thr", "rule": 31.0, "score": 1.0, "perm_p": 0.0005, "ties": ["money@t-24", "price_WHEAT", "price_WHEAT@t-1"], "loo": "16/17"}, {"turn": 150, "channel": "market", "split": "7/6", "n": 13, "parent": [4, 5, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18], "children": [[8, 9, 12, 13, 14, 15, 16], [4, 5, 6, 7, 17, 18]], "feature": "price_EGG", "kind": "eq", "rule": null, "score": 1.0, "perm_p": 0.0005, "ties": ["price_EGG", "price_EGG@t-1", "price_EGG@t-24", "price_MILK", "price_MILK@t-1", "price_MILK@t-24"], "loo": "13/13"}, {"turn": 155, "channel": "plan", "split": "4/3", "n": 7, "parent": [8, 9, 12, 13, 14, 15, 16], "children": [[8, 9, 12, 13], [14, 15, 16]], "feature": "money", "kind": "thr", "rule": 168.5, "score": 1.0, "perm_p": 0.037, "ties": ["money", "money@t-1"], "loo": "7/7"}, {"turn": 295, "channel": "plan", "split": "4/2", "n": 6, "parent": [4, 5, 6, 7, 17, 18], "children": [[4, 5, 6, 7], [17, 18]], "feature": "money", "kind": "thr", "rule": 16738.0, "score": 1.0, "perm_p": 0.1354, "ties": ["money", "money@t-1", "money@t-24", "opp_money", "opp_money@t-1", "opp_money@t-24", "price_FERTILIZER", "price_FERTILIZER@t-1", "price_FERTILIZER@t-24", "price_MELON", "price_MELON@t-1", "price_MELON@t-24", "price_MILK", "price_MILK@t-1", "price_MILK@t-24", "price_STRAWBERRY", "price_STRAWBERRY@t-1", "price_STRAWBERRY@t-24", "price_WHEAT", "price_WHEAT@t-1", "price_WOOL", "price_WOOL@t-1", "price_WOOL@t-24", "shed_WHEAT", "shed_WHEAT@t-1", "shed_WHEAT@t-24"], "loo": "6/6"}, {"turn": 306, "channel": "plan", "split": "2/2", "n": 4, "parent": [4, 5, 6, 7], "children": [[4, 5], [6, 7]], "feature": "money", "kind": "eq", "rule": null, "score": 1.0, "perm_p": 0.3443, "ties": ["money", "money@t-1", "money@t-24", "opp_money", "opp_money@t-1", "opp_money@t-24", "price_EGG", "price_EGG@t-1", "price_EGG@t-24", "price_MILK", "price_MILK@t-1", "price_MILK@t-24", "price_STRAWBERRY", "price_STRAWBERRY@t-1", "price_STRAWBERRY@t-24", "price_TOMATO", "price_TOMATO@t-1", "price_TOMATO@t-24", "unlocked", "unlocked@t-1", "unlocked@t-24"], "loo": "4/4"}, {"turn": 314, "channel": "market", "split": "2/2", "n": 4, "parent": [0, 1, 2, 3], "children": [[0, 1], [2, 3]], "feature": "money", "kind": "eq", "rule": null, "score": 1.0, "perm_p": 0.3423, "ties": ["money", "money@t-1", "money@t-24", "opp_money", "opp_money@t-1", "opp_money@t-24", "price_FERTILIZER", "price_FERTILIZER@t-24", "price_MILK", "price_MILK@t-1", "price_MILK@t-24", "price_STRAWBERRY", "price_STRAWBERRY@t-1", "price_STRAWBERRY@t-24", "price_TOMATO", "price_TOMATO@t-1", "price_TOMATO@t-24", "price_WHEAT@t-24", "price_WOOL", "price_WOOL@t-1", "price_WOOL@t-24", "unlocked", "unlocked@t-1", "unlocked@t-24"], "loo": "4/4"}, {"turn": 337, "channel": "market", "split": "2/2", "n": 4, "parent": [8, 9, 12, 13], "children": [[8, 9], [12, 13]], "feature": "money", "kind": "eq", "rule": null, "score": 1.0, "perm_p": 0.3093, "ties": ["money", "money@t-1", "money@t-24", "opp_money", "opp_money@t-1", "opp_money@t-24", "price_CARROT", "price_CARROT@t-1", "price_CARROT@t-24", "price_FERTILIZER", "price_FERTILIZER@t-1", "price_FERTILIZER@t-24", "price_MELON", "price_MELON@t-1", "price_MELON@t-24", "price_MILK", "price_MILK@t-1", "price_MILK@t-24", "price_STRAWBERRY", "price_STRAWBERRY@t-1", "price_STRAWBERRY@t-24", "price_WHEAT", "price_WHEAT@t-1", "price_WHEAT@t-24", "unlocked", "unlocked@t-1", "unlocked@t-24"], "loo": "4/4"}]''')

def _node(group):
    fk = next((f for f in MTN if f["parent"] == sorted(group)), None)
    if fk is None:
        return {"leaf": len(group)}
    return {"fork": fk, "kids": [_node(k) for k in fk["children"]]}

root = _node(list(range(19)))
LEAF_X = 400
_leafy = [0]

def _layout(node, x0):
    if "leaf" in node:
        y = _leafy[0]; _leafy[0] += 1
        node["x"], node["y"] = LEAF_X, y
        return y
    ys = [_layout(k, node["fork"]["turn"]) for k in node["kids"]]
    node["x"], node["y"] = node["fork"]["turn"], sum(ys) / len(ys)
    return node["y"]

_layout(root, 0)
fig, ax = plt.subplots(figsize=(9.2, 4.2))

def _draw(node):
    if "leaf" in node:
        ax.text(node["x"] + 6, node["y"], f"{node['leaf']} game" + ("s" if node["leaf"] != 1 else ""), va="center", fontsize=8)
        ax.scatter([node["x"]], [node["y"]], s=26, c="#4c9f70", zorder=3)
        return
    fk = node["fork"]
    sig = (fk.get("perm_p") or 1) <= 0.05
    fam = "#0f172a" if (fk.get("feature") or "").startswith("price_") else "#b45309"
    for k in node["kids"]:
        ax.plot([node["x"], k["x"]], [node["y"], k["y"]], color="#999", lw=1.0, zorder=1)
        _draw(k)
    ax.scatter([node["x"]], [node["y"]], s=120 if sig else 60,
               c=fam if sig else "white", edgecolors=fam, linewidths=1.4, zorder=3)
    _flip[0] += 1
    _dy = 10 if (sig or _flip[0] % 2) else -36
    ax.annotate(f"t={fk['turn']} (d{fk['turn'] // 24})\n{fk.get('feature', '?')}\np={fk.get('perm_p')}",
                (node["x"], node["y"]), textcoords="offset points", xytext=(-8, _dy),
                ha="right", fontsize=7)

_flip = [0]
_draw(root)
ax.axvline(72, color="#ccc", lw=0.7, ls=":")
ax.axvline(144, color="#ccc", lw=0.7, ls=":")
ax.set_xlabel("engine turn (dotted: the day-3 and day-6 unlocks; filled = p <= 0.05, navy world, amber cash)")
ax.set_yticks([])
ax.set_title("MtN's inferred decision tree, drawn as one: two world routings, then a cash branch, then noise",
             fontsize=9, loc="left")
ax.set_xlim(40, 480)
ax.margins(y=0.16)
ax.invert_yaxis()
plt.tight_layout(); plt.show()


In [ ]:
# The three blind trees plus my own reference run, embedded (2026-09-05), and
# the page's field map: every significant fork on one timeline.
TREES = _json.loads(r'''{"my own (v7.39)": [{"turn": 21, "channel": "plan", "n": 14, "split": "13/1", "feature": "money", "score": 1.0, "perm_p": 0.082, "loo": null}, {"turn": 150, "channel": "market", "n": 13, "split": "8/5", "feature": "price_WOOL", "score": 1.0, "perm_p": 0.0005, "loo": "13/13"}, {"turn": 202, "channel": "market", "n": 8, "split": "4/3/1", "feature": "price_EGG@t-24", "score": 0.893, "perm_p": 0.0145, "loo": null}, {"turn": 206, "channel": "market", "n": 4, "split": "3/1", "feature": "money", "score": 1.0, "perm_p": 0.5032, "loo": null}, {"turn": 265, "channel": "market", "n": 5, "split": "3/2", "feature": "money", "score": 1.0, "perm_p": 0.2064, "loo": "5/5"}], "MtN": [{"turn": 80, "channel": "market", "n": 19, "split": "17/2", "feature": "price_WOOL", "score": 1.0, "perm_p": 0.007, "loo": "19/19"}, {"turn": 144, "channel": "market", "n": 17, "split": "13/4", "feature": "price_WHEAT", "score": 1.0, "perm_p": 0.0005, "loo": "16/17"}, {"turn": 150, "channel": "market", "n": 13, "split": "7/6", "feature": "price_EGG", "score": 1.0, "perm_p": 0.0005, "loo": "13/13"}, {"turn": 155, "channel": "plan", "n": 7, "split": "4/3", "feature": "money", "score": 1.0, "perm_p": 0.037, "loo": "7/7"}, {"turn": 295, "channel": "plan", "n": 6, "split": "4/2", "feature": "money", "score": 1.0, "perm_p": 0.1354, "loo": "6/6"}, {"turn": 306, "channel": "plan", "n": 4, "split": "2/2", "feature": "money", "score": 1.0, "perm_p": 0.3443, "loo": "4/4"}, {"turn": 314, "channel": "market", "n": 4, "split": "2/2", "feature": "money", "score": 1.0, "perm_p": 0.3423, "loo": "4/4"}, {"turn": 337, "channel": "market", "n": 4, "split": "2/2", "feature": "money", "score": 1.0, "perm_p": 0.3093, "loo": "4/4"}], "Giulio Ravasio": [{"turn": 4, "channel": "plan", "n": 12, "split": "11/1", "feature": "opp_money", "score": 1.0, "perm_p": 0.1574, "loo": null}, {"turn": 72, "channel": "market", "n": 11, "split": "9/2", "feature": "price_CARROT", "score": 0.673, "perm_p": 1.0, "loo": "9/11"}, {"turn": 144, "channel": "market", "n": 9, "split": "7/1/1", "feature": "price_WHEAT", "score": 0.639, "perm_p": 0.4203, "loo": null}, {"turn": 155, "channel": "plan", "n": 7, "split": "5/2", "feature": "money", "score": 1.0, "perm_p": 0.09, "loo": "7/7"}, {"turn": 253, "channel": "plan", "n": 5, "split": "4/1", "feature": "money", "score": 1.0, "perm_p": 0.4083, "loo": null}, {"turn": 264, "channel": "market", "n": 4, "split": "3/1", "feature": "money", "score": 1.0, "perm_p": 0.2569, "loo": null}], "Jesse Bullard": [{"turn": 147, "channel": "market", "n": 10, "split": "9/1", "feature": "shed_CARROT", "score": 0.8, "perm_p": 1.0, "loo": null}, {"turn": 150, "channel": "market", "n": 9, "split": "6/2/1", "feature": "price_CARROT", "score": 0.944, "perm_p": 0.011, "loo": null}, {"turn": 159, "channel": "market", "n": 6, "split": "4/2", "feature": "price_WHEAT", "score": 0.8, "perm_p": 0.063, "loo": "4/4"}, {"turn": 219, "channel": "market", "n": 4, "split": "3/1", "feature": "money", "score": 1.0, "perm_p": 0.5067, "loo": null}], "kawashigi": [{"turn": 72, "channel": "market", "n": 8, "split": "6/2", "feature": "money", "score": 1.0, "perm_p": 0.028, "loo": "8/8"}, {"turn": 113, "channel": "plan", "n": 2, "split": "1/1", "feature": "unlocked", "score": 1.0, "perm_p": 1.0, "loo": null}, {"turn": 177, "channel": "plan", "n": 6, "split": "5/1", "feature": "opp_money", "score": 1.0, "perm_p": 0.1609, "loo": null}, {"turn": 264, "channel": "market", "n": 5, "split": "4/1", "feature": "money", "score": 1.0, "perm_p": 0.4178, "loo": null}, {"turn": 312, "channel": "market", "n": 4, "split": "2/2", "feature": "money", "score": 1.0, "perm_p": 0.3158, "loo": "4/4"}, {"turn": 357, "channel": "market", "n": 2, "split": "1/1", "feature": "money", "score": 1.0, "perm_p": 1.0, "loo": null}, {"turn": 384, "channel": "market", "n": 2, "split": "1/1", "feature": "money", "score": 1.0, "perm_p": 1.0, "loo": null}]}''')
def _loo_of(forks):
    h = t = 0
    for f in forks:
        if f.get("loo"):
            a, bb = map(int, f["loo"].split("/")); h += a; t += bb
    return (h, t)

LOO = {nm: _loo_of(TREES[nm]) for nm in TREES}

def family(feat):
    if not feat:
        return "other"
    if feat.startswith("price_"):
        return "world (a price indexing the shop draw)"
    if feat.startswith(("money", "opp_money")):
        return "cash"
    return "other"

FAM_COLOR = {"world (a price indexing the shop draw)": "#0f172a", "cash": "#b45309", "other": "#8a8f98"}
names = list(TREES)
fig, ax = plt.subplots(figsize=(9.2, 3.6))
for y, nm in enumerate(names):
    for f in TREES[nm]:
        p = f.get("perm_p")
        if p is None:
            continue
        fam = family(f.get("feature"))
        ax.scatter(f["turn"], y, s=30 + 10 * f["n"],
                   c=FAM_COLOR[fam] if p <= 0.05 else "none",
                   edgecolors=FAM_COLOR[fam], linewidths=1.2, zorder=3)
for x, lab in ((72, "day-3 unlock"), (144, "day-6 unlock")):
    ax.axvline(x, color="#999", lw=0.7, ls=":")
    ax.text(x, len(names) - 0.35, lab, fontsize=7, ha="center", color="#666")
ax.set_yticks(range(len(names)),
              [f"{nm}  (LOO {LOO[nm][0]}/{LOO[nm][1]})" for nm in names], fontsize=8)
ax.set_xlabel("engine turn of each fork (filled = permutation p <= 0.05; navy world, amber cash)")
ax.set_title("five trees on one timeline: the field's BRANCHERs fork at the unlocks, then on cash",
             fontsize=9, loc="left")
ax.set_xlim(0, 450)
ax.invert_yaxis()
plt.tight_layout(); plt.show()

h = sum(v[0] for v in LOO.values()); t = sum(v[1] for v in LOO.values())
print(f"page aggregate, five agents: {h}/{t} = {h / t:.1%} held-out branch prediction")


## 7. A public ground truth took the exam

While this page was in review, Thomas Tschinkel published [a notebook](https://www.kaggle.com/code/thomastschinkel/kaggriculture-public-state-router-74-5-win-rate) with the rarest thing this instrument could ask for: an agent whose decision table is written in its own docstring. Three declared forks, public state only: t=226 if a YARN_STORE has unlocked, t=360 if the carrot price is at least 42 (only under yarn), t=433 if milk market inventory is at least 10067 (only under main), plus a prefix guard and three measured repairs. Excellent, independent work: the same prefix-safety and measure-everything discipline this series argues for, arrived at on its own. It is the perfect exam, and it tested more than the tree.

**Round one: his ladder games.** His two live submissions fingerprint identically (a BRANCHER, trunk 0.755 then melt): one agent fielded twice. The tree inferred from those games says: a world fork at t=150 (the yarn worlds buy SHEEP where the others buy COW), sub-splits after, leave-one-out 24 of 24. Nothing at 226, 360 or 433. The instrument and the docstring flatly disagree.

**Round two: the disagreement is the finding.** Replaying his published code turn against turn on his own fielded games settles it: first difference at turn 0, 697 of 719 turns differ. **The published notebook is not the fielded agent**, which is entirely legitimate, a sanitized public release, and invisible to ratings, forums and docstrings alike. The inference of round one was CORRECT for the agent actually playing; behavior outranks documentation, and the instrument is how you find out which one you are looking at.

**Round three: the published code takes the exam properly.** Executed against his own recorded rivals and seeds, twenty episodes, the declared tree comes back: **the t=360 carrot fork returns with the right variable and a fitted threshold of 41.5 against his declared at-least-42, the same rule on integers, recovered blind from seven episodes** (LOO 6 of 7); the yarn fork surfaces at exactly t=226, though only as a residual split, because **his repair layer gets there first**: dead_stock sorts its extra sells by price, prices encode the world, so it separates the yarn worlds at t=151 (p = 0.0005, LOO 20 of 20), an earlier and stronger world-conditioner than the router his docstring counts; and t=433 stays out of reach at twenty episodes, the depth bound of section 9 doing exactly what it says. Repairs are decisions too, whether or not the author books them as such.

This case is kept out of the title's number, which stays pinned to the five-agent study above; folded in, the running aggregate is 187 of 195, 95.9 percent. The chart below puts the three trees of this case on one axis: what he declared, what his published code does, and what his fielded agent does.

In [ ]:
# The three trees of the Tschinkel case (measured 2026-09-06, embedded):
# what the docstring declares, what the published code does when executed on his
# own recorded rivals and seeds, and what his fielded agent does on the ladder.
TSCH = _json.loads(r'''{"fielded": [{"turn": 4, "n": 20, "split": "19/1", "feature": "money", "perm_p": 0.0865, "loo": null}, {"turn": 21, "n": 19, "split": "18/1", "feature": "money", "perm_p": 0.0545, "loo": null}, {"turn": 150, "n": 18, "split": "11/7", "feature": "price_WOOL", "perm_p": 0.0005, "loo": "18/18"}, {"turn": 202, "n": 11, "split": "7/3/1", "feature": "price_EGG@t-24", "perm_p": 0.013, "loo": null}, {"turn": 206, "n": 7, "split": "3/2/1/1", "feature": "money", "perm_p": 0.1989, "loo": null}, {"turn": 206, "n": 7, "split": "6/1", "feature": "money", "perm_p": 0.2834, "loo": null}, {"turn": 265, "n": 6, "split": "3/3", "feature": "money", "perm_p": 0.1114, "loo": "6/6"}], "published": [{"turn": 151, "n": 20, "split": "13/7", "feature": "shed_FERTILIZER", "perm_p": 0.0005, "loo": "20/20"}, {"turn": 226, "n": 13, "split": "12/1", "feature": "price_WOOL", "perm_p": 0.075, "loo": null}, {"turn": 360, "n": 7, "split": "4/3", "feature": "price_CARROT", "perm_p": 0.0595, "loo": "6/7"}, {"turn": 367, "n": 4, "split": "3/1", "feature": "money", "perm_p": 0.5022, "loo": null}, {"turn": 374, "n": 12, "split": "8/4", "feature": "money", "perm_p": 0.004, "loo": "10/12"}, {"turn": 521, "n": 8, "split": "6/1/1", "feature": "shed_WHEAT", "perm_p": 0.0325, "loo": null}, {"turn": 577, "n": 6, "split": "3/3", "feature": "money", "perm_p": 0.095, "loo": "4/6"}, {"turn": 577, "n": 4, "split": "3/1", "feature": "price_CARROT", "perm_p": 0.5017, "loo": null}]}''')
DECLARED = [(226, "YARN unlocked"), (360, "carrot >= 42"), (433, "milk inv >= 10067")]

def _loo_sum(forks):
    h = t = 0
    for f in forks:
        if f.get("loo"):
            a, bb = map(int, f["loo"].split("/")); h += a; t += bb
    return h, t

for name, forks in (("fielded agent (20 ladder games)", TSCH["fielded"]),
                    ("published code (20 games vs his recorded rivals)", TSCH["published"])):
    h, t = _loo_sum(forks)
    print(f"{name}: LOO {h}/{t}")
    for f in forks:
        print(f"   t={f['turn']:>3}  n={f['n']:>2} -> {f['split']:<9} "
              f"{f.get('feature') or '?':<18} p={f.get('perm_p')}")
    print()

fig, ax = plt.subplots(figsize=(9.2, 3.2))
rowsT = [("declared in his docstring", None), ("published code, inferred", TSCH["published"]),
         ("fielded agent, inferred", TSCH["fielded"])]
for y, (nm, forks) in enumerate(rowsT):
    if forks is None:
        for t, lab in DECLARED:
            ax.scatter(t, y, marker="D", s=70, c="#4c9f70", zorder=3)
            ax.annotate(lab, (t, y), textcoords="offset points", xytext=(0, -16),
                        ha="center", fontsize=6.6, color="#2e6b4a")
        continue
    for f in forks:
        p = f.get("perm_p")
        if p is None:
            continue
        ft = f.get("feature") or ""
        fam = "#0f172a" if ft.startswith("price_") else ("#64748B" if ft.startswith("shed_") else "#b45309")
        ax.scatter(f["turn"], y, s=26 + 8 * f["n"],
                   c=fam if p <= 0.05 else "none", edgecolors=fam, linewidths=1.1, zorder=3)
for x in (151, 226, 360, 433):
    ax.axvline(x, color="#ddd", lw=0.6, ls=":")
ax.set_yticks(range(3), [r[0] for r in rowsT], fontsize=8)
ax.set_xlabel("engine turn (filled = p <= 0.05; navy price, slate shed, amber cash; green diamonds = declared rules)")
ax.set_title("one exam, three trees: the declared table, the published code, the fielded agent",
             fontsize=9, loc="left")
ax.set_xlim(0, 620)
ax.invert_yaxis()
plt.tight_layout(); plt.show()

print("the t=360 stump fitted threshold 41.5 against the declared >= 42: the same rule on integers")


## 8. What the trees have in common

Put the five trees on one timeline and the field's BRANCHERs share a skeleton:

* **World forks sit at the unlock turns.** Every first-level routing fork of the five-agent study lands at t=72-80 or t=144-155, days 3 and 6, exactly when the town reveals its shops, and the separating feature is a product price, because the shop draw is encoded in prices. Deeper world conditioning follows within a couple of days (my own tree splits again on a price at t=202). Section 7's router aims deliberately later, t=226 and beyond, and its first behavioral world fork still lands at t=151: a price-reading repair drags even a late router to the unlock. Different teams, same reading points.
* **Cash forks sit on top.** カワシギ's whole tree is cash bands; MtN adds a cash fork after its world routing; my own finance layer edits on cash below significance. Money is the second observable this field conditions on, and so far the only other one.
* **The features that appear are the features the code consults.** In the five-agent study no fork surviving the permutation test is best explained by the rival's state or the shed; those surface only on forks the test rejects as noise. Section 7 then supplies the exception that proves the reading: its agent's dead_stock repair literally reads the shed, and a shed-read fork duly appears in its tree (t=521, p = 0.03). Either a BRANCHER reads the world and its wallet, or, when it reads something else, the tree says so.

That last sentence is a finding about the LADDER, not about the game: the game offers a rival to read, and the BRANCHERs of the top mostly do not read it.

## 9. Limits, and what a reader should not conclude

* **Depth is bought with episodes, and the LOO count says how far to trust.** Section 7's t=433 fork stays out of reach at twenty episodes because the earlier splits fragment the sample below the recursion floor; the same current number one that reads as singleton mush at ten episodes reads as a sharp tree at forty in my x-ray notebook's live section. A fork that splits two episodes one against one carries no statistics, and the instrument prints it with p = 1.0 rather than hiding it.
* **Attribution names the mechanism family and the band, not always the exact variable.** World proxies tie with each other (several prices, and the money footprint the world leaves in an opening), and own money is indistinguishable from rival money at small n because the two drift together. The ties are printed, never hidden; at these sample sizes the honest claim is often "a world read at day 6" or "a cash read near t=200", not the variable's name.
* **It reads behavior, not intention.** A repair and a router are both forks to this instrument. In section 7 that was the virtue, it surfaced a price-reading repair the author does not book as a decision; in general it means the extracted object is the BEHAVIORAL tree, which can be larger than the tree the code's author would draw.
* **It applies to BRANCHERs, and says so when it does not.** The method needs a trunk and a floor of episodes (eight at the very least, more for deep trees). On a SCRIPT there is nothing to read, on a SCHEDULER there is no trunk to anchor on, and the conditional cell in my x-ray notebook stands down with the class instead of producing noise.
* **Micro-forks need more games than routing forks.** My own finance layer is real and fires exactly where the instrument sees money splits, and fourteen episodes still cannot certify it. Absence of significance is not absence of the branch.
* **A readable tree is not a weak agent.** Reading a rival's tree tells you what it will do; beating it is a separate problem. This page measures identity, not strength, like the rest of the series.
* Replays, seeds and standings come from the public Kaggle endpoints; observations are reconstructed by replaying recorded actions through the official kaggle-environments engine (1.32.7 at the time of writing) with a bank-reproduction gate per episode. Credit to カワシギ, MtN, Giulio Ravasio, Jesse Bullard and Thomas Tschinkel, whose public games and, in the last case, publicly documented agent are the material, and to the chassis author credited in my finance notebook. Companion pages: [Which language does your agent speak?](https://www.kaggle.com/code/destbreso/which-language-does-your-agent-speak) and [X-ray your agent](https://www.kaggle.com/code/destbreso/x-ray-your-agent).